# Evaluate a trained Label-Free CBM

Run from the repo root. All logic lives in `cbm_analysis.py` — this notebook is a driver,
so **swapping models means editing one line in the Config cell.**

| Section | Call |
|---|---|
| Accuracy | `ca.evaluate(run)` |
| Compare all runs | `ca.compare(ca.find_runs())` |
| Wrong predictions | `ca.plot_wrong_predictions(run, res)` |
| Single example | `ca.explain_example(run, idx)` |
| Sankey | `ca.sankey_static(run, [class_a, class_b])` |
| Concept heatmap | `ca.concept_heatmap(run, res)` |
| Result collages | `ca.save_collages(run, res)` |
| Per-class accuracy | `ca.plot_class_accuracy(run, res)` |
| Confusion matrix | `ca.plot_confusion(run, res)` |

Everything uses matplotlib. `ca.sankey()` is an interactive plotly alternative to
`ca.sankey_static()` and is the only call that needs a non-matplotlib dependency.

In [ ]:
import os
os.environ["TORCH_DEVICE_BACKEND_AUTOLOAD"] = "0"

import torch
import cbm_analysis as ca

print("device:", "cuda" if torch.cuda.is_available() else "cpu")
print("\navailable runs:")
for d in ca.find_runs():
    print(" ", d)

## Config

Point `load_dir` at any run directory listed above. Everything below re-runs unchanged.

In [ ]:
load_dir = "saved_models/bioclip_birds525_vitb_concepts"   # <-- the only line to change
split    = "val"

run = ca.load_run(load_dir)
run

## 1. Accuracy

`evaluate` keeps per-example predictions and concept activations in dataset order,
so every plot below reuses this one forward pass.

In [ ]:
res = ca.evaluate(run, split=split)
print(res)
print("accuracy: {:.2f}%".format(res.accuracy * 100))
print("wrong:    {} of {}".format(len(res.wrong_indices), len(res.labels)))

## 2. Compare models

`summarize` reports accuracy alongside sparsity, which is the real trade-off in the
`lam` sweep: `concepts_per_class` is how many concepts a single class decision actually
depends on — the number that decides whether an explanation is readable.

In [ ]:
ca.compare(ca.find_runs(), split=split)

## 3. Wrong predictions

Each row: the misclassified image, and the concepts that pushed the model toward the
wrong class. Red bars pushed the prediction up, blue pushed it down.

In [ ]:
ca.plot_wrong_predictions(run, res, n=5, top_k=8, seed=0)

## 4. Single example

The paper's per-decision bar plot, for any index in the eval set.

In [ ]:
ca.explain_example(run, idx=20, split=split)

## 5. Sankey: concept -> class

Concept flows into two classes, weighted by the final layer, in the style of the paper's
figure. Negative weights render as `NOT <concept>`; red ribbons raise the class logit and
blue lower it. Ribbon width is `|w|`. Lower `weight_cutoff` to show more concepts.

`sankey_static` writes a publication-ready PNG and needs only matplotlib. Swap in
`ca.sankey(run, class_a=..., class_b=...)` for an interactive plotly version.

In [ ]:
print("first 10 classes:", run.classes[:10])

ca.sankey_static(run, [run.classes[0], run.classes[1]],
                 weight_cutoff=0.05, max_per_class=12,
                 save_path="figures/sankey.png")

## 6. Concept activation heatmap

Dataset-level view of which concepts fire, complementing the per-example bars.

In [ ]:
ca.concept_heatmap(run, res, n_examples=30, n_concepts=20)

## 7. Most confused class pairs

Where to point the Sankey next.

In [ ]:
for pair in ca.confused_pairs(run, res, k=10):
    print("{:4d}x  {}  ->  {}".format(pair["count"], pair["true"], pair["pred"]))

## 8. Result collages

Three sheets of 10 examples each, saved as PNGs next to the run:

- **random** — an unbiased look at typical predictions
- **confident_correct** — where the model is sure and right
- **confident_wrong** — sure and *wrong*, which is where the informative failures are

Green frame = correct, red = wrong. `p` in each caption is the softmax probability of the
predicted class.

In [ ]:
paths = ca.save_collages(run, res, n=10, seed=0)
paths

Individual sheets, if you want to tune one without regenerating all three:

In [ ]:
ca.result_collage(run, res, mode="confident_wrong", n=10)

## 9. Per-class accuracy

Left: how per-class accuracy is distributed across all classes. Right: the weakest classes
by name, with `correct/support` so you can see how much evidence each number rests on.

**Read the support numbers before trusting the ranking.** Birds525 ships ~5 validation
images per class, so per-class accuracy can only take 6 distinct values and the "worst"
list is dominated by sampling noise. It is a shortlist of classes to inspect, not a
reliable ordering.

In [ ]:
rows = ca.plot_class_accuracy(run, res, worst_k=25,
                              save_path="figures/class_accuracy.png")

print("\nworst 10:")
for r in rows[:10]:
    print("  {:5.1f}%  {:3d}/{:<3d}  {}".format(
        r["accuracy"] * 100, r["correct"], r["support"], r["class"]))

## 10. Confusion matrix

The full matrix is 525x525 and almost entirely zeros, so this restricts rows to the
weakest classes and columns to the predictions they actually receive. Green boxes mark the
correct cell — a row with an empty green box and mass elsewhere is a class the model
consistently mistakes for something specific.

Those are the pairs worth pointing the Sankey at in section 5: if two species share most of
their top concepts, that is the bottleneck failing to separate them.

In [ ]:
mat = ca.plot_confusion(run, res, worst_k=25,
                        save_path="figures/confusion.png")

## 11. Cross-model qualitative comparison

One figure, four rows, each a different story about where the backbones diverge:

| Row | What it shows |
|---|---|
| **models disagree most** | the image the five models split on hardest |
| **strong right, domain wrong** | where generic pretraining beats domain pretraining |
| **domain right, strong wrong** | the reverse — the counter-cases |
| **every model wrong** | genuine ambiguity or label noise |

Both "wins" rows are deliberate. A figure showing only cases where your preferred model
wins is cherry-picking, and the image counts per row (in the left labels) tell you how
often each situation actually occurs — that count is the real result, not the single
image chosen to illustrate it.

`evaluate_many` loads and releases one backbone at a time, so five ViT-B/16 models never
sit on the GPU together. `keep_concept_acts=True` costs ~20MB per run and is what lets
each cell show the concepts behind that model's prediction.

In [ ]:
comparison_runs = [
    "saved_models/in21k_birds525",
    "saved_models/bioclip_birds525_vitb_concepts",
    "saved_models/clip_vitb16_birds525",
    "saved_models/dino_birds525",
    "saved_models/rn50_birds525",
]

outs = ca.evaluate_many(comparison_runs, split=split, keep_concept_acts=True)

In [ ]:
ca.story_figure(outs, split=split, top_concepts=2,
                save_path="figures/qualitative_comparison.png")

`strong` defaults to the most accurate run and `domain` to the BioCLIP one. Override
either when you want a specific contrast:

In [ ]:
picks = ca.story_examples(outs, strong="in21k_birds525",
                          domain="bioclip_birds525_vitb_concepts")
print("strong:", picks.pop("_strong"), "| domain:", picks.pop("_domain"))
for mode, info in picks.items():
    print("  {:20s} {:4d} images   example #{}".format(mode, info["count"], info["index"]))